# Multi-Objective & Semantic Re-Ranking Engine

Build a three-stage re-ranking pipeline: score blending, vectorized MMR
diversity selection, and post-hoc creator fairness enforcement, with
results tracked to MLflow.

## Setup: Load the candidate dataset

Run this cell first. The `embedding_vector` column is stored as a
JSON-encoded string in the CSV (a CSV cell can't natively hold a Python
list), so it needs to be parsed back into real numeric values before you
can build a NumPy matrix from it.

In [ ]:
import json
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

candidates_df = pd.read_csv("path/candidates_embeddings.csv")

# Parse the JSON-encoded embedding strings back into real lists of floats
candidates_df["embedding_vector"] = candidates_df["embedding_vector"].apply(json.loads)

# Build a proper NumPy matrix from the parsed embeddings
embeddings_matrix = np.stack(candidates_df["embedding_vector"].values)

print(f"Loaded {len(candidates_df)} candidates")
print(f"Embeddings matrix shape: {embeddings_matrix.shape}")
print(candidates_df[["item_id", "saves_score", "dwell_score", "shares_score", "is_new_creator"]].head())

## Task 1: Execute Composite Score Blending (10 min)

Combine the separate prediction scores for Saves, Dwell Time, and Private
Shares into a single `base_score`, using the provided enterprise utility
weights.

In [ ]:
CONFIG_WEIGHTS = {
    "saves": 0.5,
    "dwell": 0.3,
    "shares": 0.2
}

def blend_multitask_scores(df, weights):
    """Normalizes and computes weighted sum of multi-task logits."""
    # --- Your code here ---
    # Calculate the weighted score and add it to the DataFrame as a new
    # column named 'base_score'. Use weights['saves'], weights['dwell'],
    # and weights['shares'] against the df's saves_score, dwell_score,
    # and shares_score columns.


    return df
    # --- End of your code ---

In [ ]:
# Run Task 1
candidates_df = blend_multitask_scores(candidates_df, CONFIG_WEIGHTS)

# Verify the output
print(candidates_df[["item_id", "base_score"]].head())
print(candidates_df["base_score"].describe())

**Success Checklist**
- Function correctly calculates the weighted sum of the three score columns.
- A new column named `base_score` is successfully added to the DataFrame.
- The function returns the modified DataFrame.

## Task 2: Implement the Vectorized MMR Diversity Loop (15 min)

Select a diverse top-10 list using Maximal Marginal Relevance (MMR). The
similarity computation at each step must be vectorized across all
remaining candidates simultaneously — the outer selection loop itself is
expected and unavoidable in MMR, but there should be no inner
element-wise iteration.

In [ ]:
def apply_mmr(candidates_df, embeddings_matrix, lambda_param=0.5, top_k=10):
    """Selects top_k candidates using a highly optimized, vectorized MMR selection."""
    selected_indices = []
    candidate_indices = list(range(len(candidates_df)))

    # Select first item purely by maximum base relevance
    first_idx = candidates_df['base_score'].idxmax()
    selected_indices.append(first_idx)
    candidate_indices.remove(first_idx)

    while len(selected_indices) < top_k and candidate_indices:
        selected_embeddings = embeddings_matrix[selected_indices]
        remaining_embeddings = embeddings_matrix[candidate_indices]

        # Vectorized similarity matrix calculation
        sim_matrix = cosine_similarity(remaining_embeddings, selected_embeddings)
        max_sims = np.max(sim_matrix, axis=1)

        base_scores = candidates_df.loc[candidate_indices, 'base_score'].values

        # --- Your code here ---
        # Implement the vectorized MMR Core Formula to calculate mmr_scores
        # for all remaining candidates at once.
        # Formula: mmr = (1 - lambda) * relevance - (lambda) * similarity


        # --- End of your code ---

        best_remaining_idx = np.argmax(mmr_scores)
        best_idx = candidate_indices[best_remaining_idx]

        selected_indices.append(best_idx)
        candidate_indices.remove(best_idx)

    return candidates_df.loc[selected_indices].copy()

In [ ]:
# Run Task 2
mmr_top10 = apply_mmr(candidates_df, embeddings_matrix, lambda_param=0.5, top_k=10)
print(mmr_top10[["item_id", "base_score", "is_new_creator"]])

# Try lambda_param=0 (pure relevance, no diversity penalty) and
# lambda_param=1 (pure diversity, relevance ignored) to see how the
# selection changes.

**Success Checklist**
- The MMR formula `(1 - lambda) * relevance - (lambda) * similarity` is correctly implemented.
- The per-step similarity calculation is fully vectorized across all remaining candidates.
- The function returns a DataFrame containing exactly `top_k` items.

## Task 3: Integrate the Creator Parity Filter (13 min)

Check the MMR-ranked top-10 list. If it contains fewer than 2 new-creator
items, swap out the lowest-scoring established items to meet the 20%
quota.

In [ ]:
def enforce_fairness(selected_df, all_candidates_df, min_new_creators=2):
    """Applies a post-hoc exposure quota override to protect creator equity."""
    new_creator_count = (selected_df['is_new_creator'] == True).sum()

    if new_creator_count >= min_new_creators:
        return selected_df

    # --- Your code here ---
    # 1. Calculate the 'deficit' of new creators needed.
    # 2. Find the best 'unused_new' creator items from 'all_candidates_df'
    #    that are not already in 'selected_df' (sort by base_score, descending).
    # 3. Identify the 'established_to_drop' items from the tail of
    #    'selected_df' (the lowest-scoring established items).
    # 4. Create 'final_df' by dropping the identified items and
    #    concatenating the new ones.


    # --- End of your code ---
    return final_df

In [ ]:
# Test 1: a list that already meets the quota (should return unchanged)
sample_ok = mmr_top10.copy()
sample_ok.loc[sample_ok.index[0], "is_new_creator"] = True
sample_ok.loc[sample_ok.index[1], "is_new_creator"] = True
result_ok = enforce_fairness(sample_ok, candidates_df, min_new_creators=2)
print("Test 1 (already meets quota):", (result_ok["is_new_creator"] == True).sum(), "new creators")

# Test 2: the real MMR output, which may have zero new creators
final_list = enforce_fairness(mmr_top10, candidates_df, min_new_creators=2)
print("Test 2 (real MMR output):", (final_list["is_new_creator"] == True).sum(), "new creators")
print(final_list[["item_id", "base_score", "is_new_creator"]])

**Success Checklist**
- The function correctly calculates the deficit of new creators.
- The function returns the original DataFrame unchanged if the quota is already met.
- If a deficit exists, the lowest-ranked established items are correctly swapped for the highest-ranked available new creator items.
- The final returned DataFrame always contains `top_k` items.

## Task 4: Programmatically Instrument MLflow Logs (15 min)

Wrap the full pipeline in an MLflow run, log the `lambda_param`, compute
and log three metrics (NDCG@10, diversity entropy, new creator ratio),
and log a trade-off plot as an artifact.

In [ ]:
import mlflow
import matplotlib.pyplot as plt

def calculate_metrics(final_list, all_candidates_df, embeddings_matrix, all_indices_map, n_clusters=8):
    """Computes precision (NDCG@10), diversity entropy, and new creator exposure ratio."""
    # --- Your code here ---
    # 1. NDCG@10: use base_score as the graded relevance value. Compute DCG
    #    on the actual order of final_list; compute IDCG on the ideal order
    #    (the top 10 items from all_candidates_df sorted purely by
    #    base_score). NDCG@10 = DCG / IDCG.
    # 2. Diversity entropy: cluster all_candidates_df's embeddings (e.g.
    #    with sklearn's KMeans), then compute the normalized Shannon
    #    entropy of final_list's distribution across those clusters.
    # 3. new_creator_ratio: the fraction of final_list that is a new creator.
    #
    # Return a dict: {"ndcg_10": ..., "diversity_entropy": ..., "new_creator_ratio": ...}


    # --- End of your code ---
    pass

In [ ]:
# --- Your code here ---
# Wrap your entire pipeline execution and logging in this block.
#
# Example structure:
# with mlflow.start_run():
#     mlflow.log_param("mmr_lambda", lambda_val)
#
#     blended_df = blend_multitask_scores(candidates_df, CONFIG_WEIGHTS)
#     mmr_selected_df = apply_mmr(blended_df, embeddings_matrix, lambda_param=lambda_val)
#     final_ranked_list = enforce_fairness(mmr_selected_df, blended_df)
#
#     all_indices_map = {idx: pos for pos, idx in enumerate(blended_df.index)}
#     metrics = calculate_metrics(final_ranked_list, blended_df, embeddings_matrix, all_indices_map)
#     mlflow.log_metrics(metrics)
#
#     fig, ax = plt.subplots()
#     ax.scatter(metrics["diversity_entropy"], metrics["ndcg_10"])
#     ax.set(xlabel="Diversity Entropy", ylabel="NDCG@10", title="Performance Trade-off")
#     fig.savefig("tradeoff_plot.png")
#     mlflow.log_artifact("tradeoff_plot.png")


# --- End of your code ---

**Success Checklist**
- The entire pipeline execution is wrapped within an `mlflow.start_run()` context.
- The MMR `lambda_param` is logged as a parameter.
- Three distinct metrics (NDCG@10, Diversity Entropy, New Creator Ratio) are logged.
- At least one artifact (e.g., a trade-off plot image file) is logged to the run.

## Common Mistakes to Avoid

- **Inefficient loops**: calculating MMR scores one-by-one instead of vectorizing across all remaining candidates in a single expression per step.
- **Incorrect indexing**: forgetting to remove selected indices from the candidate pool in the MMR loop, causing infinite loops or repeated selections.
- **Stateful DataFrame modification**: modifying a DataFrame without returning it, or relying on global state — each function should accept a DataFrame and return one.
- **Authentication errors**: forgetting to set the Azure ML tracking URI correctly, causing MLflow to log locally instead of to the shared workspace registry.
- **Embedding parsing**: forgetting to parse the `embedding_vector` column with `json.loads()` before calling `np.stack()` — skipping this produces a silently wrong array of strings rather than a numeric matrix.